# 🤖 vla-hands — Quick Start (Colab / Kaggle)

This notebook gives a VLM **hands** by grafting lightweight action heads onto its
frozen hidden states — turning it into a VLA (Vision-Language-Action) model.

We'll train four appendage types on tiny environments:
| Appendage | Environment | Task |
|-----------|-------------|------|
| 🕹️ Joystick | Target Nav / Spaceship | Continuous 2D navigation |
| 🎮 D-pad | Grid World / Maze | Discrete directional movement |
| 🔘 Button | Color Press | Press when circle matches target color |
| 🎛️ MultiButton | MCQ | Answer multiple-choice visual questions |

**Runtime:** T4 GPU on Colab Free / P100 on Kaggle  
**Model:** `HuggingFaceTB/SmolVLM-256M-Instruct` (~1 GB, fits on T4 easily)  
**Training time:** ~10–15 min for a quick demo run

## 1 · Setup

In [ ]:
# Install the package directly from GitHub
# (change the URL/branch if you have a fork)
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza

# Or, if you've cloned it locally (e.g. on Kaggle with a dataset):
# import sys; sys.path.insert(0, '/kaggle/input/vla-hands/project-h')

print('✅ vla-hands installed')

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from PIL import Image

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Visualise the Environments

Before training anything, let's see what each environment looks like and verify
the expert policy works as expected.

In [ ]:
from vla_hands import (
    TargetNavEnvironment,
    SpaceshipNavEnvironment,
    GridWorldEnvironment,
    MazeEnvironment,
    ButtonPressEnvironment,
    MCQButtonEnvironment,
)

envs = {
    'Target Nav (joystick)': TargetNavEnvironment(width=200, height=200),
    'Spaceship (joystick)':  SpaceshipNavEnvironment(width=200, height=200),
    'Grid World (d-pad)':    GridWorldEnvironment(grid_size=6),
    'Maze (d-pad)':          MazeEnvironment(rows=6, cols=6),
    'Color Press (button)':  ButtonPressEnvironment(width=200, height=200),
    'MCQ (multi-button)':    MCQButtonEnvironment(width=240, height=200),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, env) in zip(axes.flat, envs.items()):
    obs = env.reset(seed=42)
    ax.imshow(obs)
    ax.set_title(name, fontsize=11)
    ax.axis('off')
plt.suptitle('VLA-Hands: Available Training Environments', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Roll out the expert policy for a few steps and render the trajectory
def render_expert_rollout(env, n_steps=12, cols=6, seed=7):
    obs = env.reset(seed=seed)
    frames = [obs]
    for _ in range(n_steps - 1):
        action = env.expert_action()
        result = env.step(action)
        frames.append(result.observation)
        if result.done:
            break

    rows = (len(frames) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
    axes = np.array(axes).flat
    for ax, frame in zip(axes, frames):
        ax.imshow(frame)
        ax.axis('off')
    for ax in list(axes)[len(frames):]:
        ax.set_visible(False)
    plt.suptitle(f'{type(env).__name__} — Expert Rollout')
    plt.tight_layout()
    plt.show()

render_expert_rollout(TargetNavEnvironment(224, 224), n_steps=8)

In [ ]:
render_expert_rollout(MazeEnvironment(rows=5, cols=5), n_steps=20, cols=5)

In [ ]:
# Expert baseline — the theoretical ceiling for each environment
from vla_hands import run_expert_baseline

print('Expert baselines (20 episodes each):')
for name, env in [
    ('TargetNav', TargetNavEnvironment()),
    ('Spaceship', SpaceshipNavEnvironment()),
    ('GridWorld', GridWorldEnvironment()),
    ('Maze 7x7', MazeEnvironment()),
    ('ColorPress', ButtonPressEnvironment()),
    ('MCQ', MCQButtonEnvironment()),
]:
    r = run_expert_baseline(env, n_episodes=20)
    print(f'  {name:12s} success={r.success_rate:.0%}  reward={r.mean_reward:+.1f}')

## 3 · Load the VLM

We use **SmolVLM-256M-Instruct** — a tiny but capable vision-language model  
that fits in ~2 GB of VRAM and loads in under a minute.

In [ ]:
from transformers import AutoModelForVision2Seq, AutoProcessor

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
# Alternatives (larger, better visual understanding):
# MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'  # ~2.5 GB
# MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'       # ~4 GB (2B params)

print(f'Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,   # float16 if VRAM is tight
)

n_params = sum(p.numel() for p in vlm.parameters())
hidden_dim = vlm.config.hidden_size
print(f'Parameters:  {n_params/1e6:.0f}M')
print(f'Hidden dim:  {hidden_dim}')
print(f'Model type:  {type(vlm).__name__}')

## 4 · Create Grafts

Each graft pairs a **VLM** with an **appendage** (action head).  
The VLM backbone is frozen to start — we only train the tiny MLP.

In [ ]:
from vla_hands import (
    VLAGraft, GraftConfig,
    JoystickAppendage,
    DPadAppendage,
    ButtonAppendage,
    MultiButtonAppendage,
)

# Shared config: extract features from last token position
cfg = GraftConfig(feature_extraction='last')

# Four grafts sharing the same frozen VLM backbone
joystick_graft = VLAGraft(vlm=vlm, appendage=JoystickAppendage(hidden_dim), config=cfg)
dpad_graft     = VLAGraft(vlm=vlm, appendage=DPadAppendage(hidden_dim),     config=cfg)
button_graft   = VLAGraft(vlm=vlm, appendage=ButtonAppendage(hidden_dim),   config=cfg)
mcq_graft      = VLAGraft(vlm=vlm, appendage=MultiButtonAppendage(hidden_dim, n_buttons=4,
                                              labels=['A','B','C','D']), config=cfg)

print(joystick_graft)
print(f'\nAppendage sizes:')
for name, g in [('joystick', joystick_graft), ('dpad', dpad_graft),
                ('button', button_graft), ('mcq_4btn', mcq_graft)]:
    n = g.appendage.num_parameters()
    print(f'  {name:12s} {n:,} params  ({n/1e3:.0f}K)')

## 5 · Train

We run a BC (Behavioral Cloning) warm-start followed by brief RL fine-tuning.

Training uses the **`QUICK_CURRICULUM`** (appendage weights only, VLM frozen)
so it's fast enough to run in a free Colab session.

To get serious performance: switch to `DEFAULT_CURRICULUM` and increase steps.

In [ ]:
from vla_hands import TrainingCurriculum, CurriculumConfig
from vla_hands import QUICK_CURRICULUM

def quick_train(graft, env, bc_steps=200, rl_steps=50, tag=''):
    """Train one graft+environment pair and return metrics."""
    config = CurriculumConfig(
        bc_steps=bc_steps,
        rl_steps=rl_steps,
        device=device,
        save_dir=f'checkpoints/{tag or type(env).__name__}',
        freezing_stages=QUICK_CURRICULUM,   # keep VLM fully frozen for speed
        eval_every=bc_steps // 4,
        log_every=bc_steps // 10,
        eval_episodes=5,
    )
    curriculum = TrainingCurriculum(graft, processor, env, config)
    return curriculum.run()

print('Training helpers ready.')

In [ ]:
# ── Joystick: Target Navigation ──────────────────────────────────────────────
print('='*60)
print('Training: Joystick → Target Navigation')
print('='*60)

target_env = TargetNavEnvironment(width=224, height=224, max_steps=80)
joystick_metrics = quick_train(joystick_graft, target_env, bc_steps=300, rl_steps=100,
                               tag='joystick_target')

In [ ]:
# ── D-pad: Grid World ────────────────────────────────────────────────────────
print('='*60)
print('Training: D-pad → Grid World')
print('='*60)

grid_env = GridWorldEnvironment(grid_size=6)
dpad_metrics = quick_train(dpad_graft, grid_env, bc_steps=300, rl_steps=100,
                           tag='dpad_grid')

In [ ]:
# ── Button: Color Recognition ────────────────────────────────────────────────
print('='*60)
print('Training: Button → Color Press')
print('='*60)

color_env = ButtonPressEnvironment(difficulty='easy')
button_metrics = quick_train(button_graft, color_env, bc_steps=200, rl_steps=0,
                             tag='button_color')

In [ ]:
# ── MultiButton: MCQ ─────────────────────────────────────────────────────────
print('='*60)
print('Training: MultiButton → Multiple Choice Questions')
print('='*60)

mcq_env = MCQButtonEnvironment(question_type='dots')
mcq_metrics = quick_train(mcq_graft, mcq_env, bc_steps=200, rl_steps=0,
                          tag='mcq_dots')

## 6 · Benchmark & Visualise Results

In [ ]:
def plot_metrics(metrics_dict, key='bc/loss', title='Training Loss'):
    fig, axes = plt.subplots(1, len(metrics_dict), figsize=(5*len(metrics_dict), 4))
    if len(metrics_dict) == 1:
        axes = [axes]
    for ax, (name, metrics) in zip(axes, metrics_dict.items()):
        bc_metrics = metrics.get('bc', metrics)
        steps = [m['step'] for m in bc_metrics if key in m]
        vals  = [m[key]  for m in bc_metrics if key in m]
        ax.plot(steps, vals, linewidth=2)
        ax.set_title(name)
        ax.set_xlabel('Step')
        ax.set_ylabel(key)
        ax.grid(alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_metrics({
    'Joystick': joystick_metrics,
    'D-pad':    dpad_metrics,
    'Button':   button_metrics,
    'MCQ':      mcq_metrics,
}, key='bc/loss', title='BC Training Loss')

In [ ]:
from vla_hands import BenchmarkSuite

results = {}
benchmark_pairs = [
    ('Joystick',  joystick_graft, TargetNavEnvironment()),
    ('D-pad',     dpad_graft,     GridWorldEnvironment(grid_size=6)),
    ('Button',    button_graft,   ButtonPressEnvironment()),
    ('MCQ',       mcq_graft,      MCQButtonEnvironment()),
]

for name, graft, env in benchmark_pairs:
    suite = BenchmarkSuite(graft, processor, [env], device=device)
    r = suite.run_benchmark(env, n_episodes=15)
    results[name] = r
    print(f'{name:12s}  success={r.success_rate:.0%}  reward={r.mean_reward:+.1f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

names    = list(results.keys())
sr_vals  = [results[n].success_rate * 100 for n in names]
rw_vals  = [results[n].mean_reward for n in names]

bars = ax1.bar(names, sr_vals, color=['#4C9BE8','#5BC46A','#E8804C','#C45BE8'])
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('Success Rate per Appendage')
ax1.set_ylim(0, 110)
for bar, val in zip(bars, sr_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=11)

ax2.bar(names, rw_vals, color=['#4C9BE8','#5BC46A','#E8804C','#C45BE8'])
ax2.set_ylabel('Mean Episode Reward')
ax2.set_title('Mean Reward per Appendage')
ax2.axhline(0, color='gray', linewidth=0.5)

plt.suptitle('vla-hands Benchmark Results', fontsize=13)
plt.tight_layout()
plt.show()

## 7 · Interactive Inference

Run the trained grafts on fresh episodes and see what they predict.

In [ ]:
def run_inference_demo(graft, env, n_steps=8, seed=99):
    """Run a few steps of inference and display side-by-side with predicted action."""
    from vla_hands.training.trainer import _preprocess

    graft.eval()
    obs = env.reset(seed=seed)
    frames, actions = [obs], []

    for step in range(n_steps):
        with torch.no_grad():
            inputs = _preprocess(processor, obs, env.prompt, device)
            out = graft(**inputs)
            raw_action = out['action']

        decoded = graft.appendage.decode(raw_action)
        actions.append(decoded)

        # Get concrete action for env.step
        if hasattr(graft.appendage, 'argmax'):
            action_val = int(graft.appendage.argmax(raw_action).item())
        else:
            action_val = raw_action.squeeze(0).cpu().tolist()

        result = env.step(action_val)
        obs = result.observation
        frames.append(obs)
        if result.done:
            print(f'  Episode ended at step {step+1} — success={result.info.get("success")}')
            break

    # Display
    fig, axes = plt.subplots(1, len(frames), figsize=(3*len(frames), 3))
    for i, (ax, frame) in enumerate(zip(axes, frames)):
        ax.imshow(frame)
        ax.axis('off')
        if i < len(actions):
            ax.set_title(str(actions[i])[:18], fontsize=7)
    plt.suptitle(f'{type(graft.appendage).__name__} on {type(env).__name__}')
    plt.tight_layout()
    plt.show()


print('Joystick inference:')
run_inference_demo(joystick_graft, TargetNavEnvironment(), n_steps=8)

print('D-pad inference:')
run_inference_demo(dpad_graft, GridWorldEnvironment(grid_size=6), n_steps=10)

print('Button inference (should press ~half the time):')
run_inference_demo(button_graft, ButtonPressEnvironment(), n_steps=4)

print('MCQ inference:')
run_inference_demo(mcq_graft, MCQButtonEnvironment(), n_steps=3)

## 8 · Save & Share

Checkpoints are tiny: only the appendage MLP weights (~100 KB–1 MB each).
The VLM backbone is loaded separately from HuggingFace Hub.

In [ ]:
import os, json

save_dir = 'vla_hands_checkpoints'
os.makedirs(save_dir, exist_ok=True)

for name, graft in [
    ('joystick', joystick_graft),
    ('dpad',     dpad_graft),
    ('button',   button_graft),
    ('mcq',      mcq_graft),
]:
    path = f'{save_dir}/{name}'
    graft.save(path)
    # Inspect saved files
    files = os.listdir(path)
    sizes = {f: os.path.getsize(f'{path}/{f}') for f in files}
    print(f'{name}: {files} — {sum(sizes.values())/1024:.1f} KB total')

print('\nCheckpoint structure for joystick:')
with open(f'{save_dir}/joystick/graft_config.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Reload a saved appendage (demonstrates the save/load round-trip)
reloaded_graft = VLAGraft(
    vlm=vlm,
    appendage=JoystickAppendage(hidden_dim),
    config=cfg,
)
reloaded_graft.load_appendage(f'{save_dir}/joystick')
print('Reload successful!')

# Quick sanity check: forward pass
test_env = TargetNavEnvironment()
obs = test_env.reset(seed=1)
inputs = processor(images=obs, text=test_env.prompt, return_tensors='pt')
with torch.no_grad():
    action = reloaded_graft.predict_action(**{k: v.to(device) for k, v in inputs.items()})
decoded = reloaded_graft.appendage.decode(action)
print(f'Test action: {decoded}')

In [ ]:
# ── Upload to HuggingFace Hub (optional) ─────────────────────────────────────
# pip install -q huggingface_hub
# from huggingface_hub import HfApi
#
# api = HfApi()
# api.upload_folder(
#     folder_path=save_dir,
#     repo_id='your-username/vla-hands-checkpoints',
#     repo_type='model',
# )
print('Uncomment the block above to push checkpoints to HuggingFace Hub.')

## 9 · Advanced: Train the Harder Environments

In [ ]:
# Spaceship has momentum — harder for BC but more realistic
# Needs more steps than basic target nav
ship_graft = VLAGraft(vlm=vlm, appendage=JoystickAppendage(hidden_dim), config=cfg)
ship_env   = SpaceshipNavEnvironment(width=224, height=224, max_steps=150)

ship_metrics = quick_train(ship_graft, ship_env, bc_steps=400, rl_steps=150,
                           tag='joystick_spaceship')

suite = BenchmarkSuite(ship_graft, processor, [ship_env], device=device)
r = suite.run_benchmark(ship_env, n_episodes=15)
print(r)

In [ ]:
# Maze needs backtracking — much harder than grid world
maze_graft = VLAGraft(vlm=vlm, appendage=DPadAppendage(hidden_dim), config=cfg)
maze_env   = MazeEnvironment(rows=5, cols=5)   # start small

maze_metrics = quick_train(maze_graft, maze_env, bc_steps=500, rl_steps=200,
                           tag='dpad_maze')

suite = BenchmarkSuite(maze_graft, processor, [maze_env], device=device)
r = suite.run_benchmark(maze_env, n_episodes=15)
print(r)

In [ ]:
# Full curriculum: gradually unfreeze VLM layers as training progresses
# Much better final performance, but slower
from vla_hands import DEFAULT_CURRICULUM

full_joystick_graft = VLAGraft(vlm=vlm, appendage=JoystickAppendage(hidden_dim), config=cfg)
config = CurriculumConfig(
    bc_steps=1500,
    rl_steps=500,
    device=device,
    freezing_stages=DEFAULT_CURRICULUM,   # gradually unfreeze up to last 6 layers
    save_dir='checkpoints/joystick_full_curriculum',
    eval_every=300,
    log_every=100,
)
curriculum = TrainingCurriculum(full_joystick_graft, processor, TargetNavEnvironment(), config)
# curriculum.run()   # uncomment for full training (takes ~30 min on T4)
print('Uncomment curriculum.run() for full training with layer unfreezing.')

## 10 · What's Next?

```
Appendages to add:     pointer (2D screen tap), slider (1D), gyro (3-axis)
Environments to add:   snake game, berry collection (nav+button), camera rotate
Training ideas:        multi-task (train one graft on multiple envs simultaneously)
                       meta-learning (fast adaptation to new environments)
                       self-supervised (reward from language model's own confidence)
Model ideas:           CompositeGraft (joystick + button simultaneously)
                       Shared backbone across multiple appendages
Deployment:            ONNX export, TorchScript, edge inference
```

**Contribute** at the GitHub repo. PRs welcome for new appendages, environments, and training ideas!

---
_vla-hands — MIT License_